# Prajna Training — Offline Distillation

**E4B Teacher → E2B Student with Full CRN Architecture**

1. Loads E4B teacher (4-bit) → generates training data
2. Creates E2B student with 4 CRN pillars via hooks
3. Trains via offline distillation with contrastive loss
4. Saves to Google Drive (checkpoints + memory state)

**CRN Pillars:**
- Resonance Attention: Frequency-band gating on hidden states
- Episodic Memory: Cross-session persistent memory (save/load)
- Reflective Loop: Contrastive-trained self-correction (16 directions)
- Skill Composition: 64 low-rank skills with load balancing

**Required:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Install & Verify
!pip install -q torch transformers accelerate bitsandbytes einops

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ERROR: No GPU! Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2: Mount Drive & Create Dirs
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/prajna/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/prajna/data', exist_ok=True)
print('Drive mounted, directories created')

In [ ]:
# Cell 3: Full CRN Implementations (from prajna-toy-validation)
# These are the ACTUAL implementations, not simplified versions.

import torch.nn as nn
import torch.nn.functional as F
from einops import einsum
import json

# ─── Pillar 1: Resonance Attention ───────────────────────────────────────
# Frequency-band gating: groups tokens by cognitive frequency,
# applies band-specific transformations.

class ResonanceAttention(nn.Module):
    """
    Frequency-modulated attention with learned cognitive bands.
    Tokens are projected to frequency space, top-k frequencies selected,
    and band-specific transformations applied.
    """

    COGNITIVE_MODES = [
        "DEFINE", "EXPLAIN", "ARGUE", "CALCULATE",
        "HYPOTHESIZE", "EVIDENCE", "SUMMARIZE", "QUESTION",
        "REFLECT", "CORRECT", "TOOL_CALL", "TOOL_RESULT",
        "NARRATE", "CLASSIFY", "COMPARE", "GENERATE"
    ]

    def __init__(self, d_model, num_heads=4, num_frequencies=16, top_k=4):
        super().__init__()
        self.num_heads = num_heads
        self.num_frequencies = num_frequencies
        self.top_k = top_k
        self.head_dim = d_model // num_heads

        # Per-head frequency projections
        self.freq_q = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.freq_k = nn.Linear(d_model, num_heads * num_frequencies, bias=False)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        # Learned transition graph: which frequencies can attend to which
        self.transition_graph = nn.Parameter(
            torch.randn(num_frequencies, num_frequencies) * 0.01
        )

        # Frequency assignment (which cognitive mode each frequency represents)
        self.freq_assignment = nn.Parameter(
            torch.randn(num_frequencies, num_frequencies) * 0.01
        )

    def forward(self, x, return_freq_info=False):
        B, T, D = x.shape

        # Project to frequency space: [B, T, H, F]
        q = self.freq_q(x).view(B, T, self.num_heads, self.num_frequencies)
        k = self.freq_k(x).view(B, T, self.num_heads, self.num_frequencies)

        # Compute frequency assignment scores per token
        freq_scores = F.softmax(q, dim=-1)  # [B, T, H, F]

        # Select top-k frequencies per token (TRULY SPARSE - no n×n matrix)
        top_k = min(self.top_k, self.num_frequencies)
        top_freq_vals, top_freq_idx = freq_scores.topk(top_k, dim=-1)  # [B, T, H, top_k]
        top_freq_vals = top_freq_vals / (top_freq_vals.sum(dim=-1, keepdim=True) + 1e-8)

        # Compute value
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim)

        # Sparse attention: only compute within compatible frequency groups
        out = torch.zeros_like(v)

        for f_idx in range(self.num_frequencies):
            # Find tokens where this frequency is in their top-k
            mask = (top_freq_idx == f_idx).any(dim=-1)  # [B, T, H]

            if mask.sum() == 0:
                continue

            # Get the weight for this frequency across all tokens
            freq_weight = torch.zeros(B, T, self.num_heads, device=x.device)
            for k_idx in range(top_k):
                match = (top_freq_idx[:, :, :, k_idx] == f_idx)
                freq_weight += match.float() * top_freq_vals[:, :, :, k_idx]

            # Compute attention ONLY among tokens sharing this frequency
            q_f = q[:, :, :, f_idx]  # [B, T, H]
            k_f = k[:, :, :, f_idx]  # [B, T, H]

            # Attention scores (sparse: only among masked tokens)
            attn_scores = einsum(q_f, k_f, 'b i h, b j h -> b h i j')
            attn_scores = attn_scores / (self.head_dim ** 0.5)

            # Mask out tokens not in this frequency group
            attn_mask = mask.unsqueeze(2) * mask.unsqueeze(1)  # [B, T, T, H]
            attn_mask = attn_mask.permute(0, 3, 1, 2)  # [B, H, T, T]
            attn_scores = attn_scores.masked_fill(~attn_mask.bool(), float('-inf'))

            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_weights = attn_weights.nan_to_num(0.0)

            # Weighted sum of values
            attn_out = einsum(attn_weights, v, 'b h i j, b j h d -> b i h d')

            # Apply frequency weight
            out += freq_weight.unsqueeze(-1) * attn_out

        # Reshape and project
        out = out.reshape(B, T, D)
        out = self.out_proj(out)

        if return_freq_info:
            dominant_freq = top_freq_idx[:, :, :, 0]  # [B, T, H]
            return out, {
                "dominant_frequency": dominant_freq,
                "frequency_scores": freq_scores,
                "top_frequencies": top_freq_idx,
            }

        return out


# ─── Pillar 2: Episodic Memory ───────────────────────────────────────────
# Runtime state (NOT nn.Parameter), cross-session persistence via save/load.
# Uses learned gates for read/write and compression for efficiency.

class EpisodicMemory:
    """
    Runtime episodic memory decoupled from model parameters.
    Memory is NOT an nn.Parameter. It's a runtime state tensor
    that can be saved/loaded independently of model weights.
    """

    def __init__(self, d_model, mem_size=512, mem_dim=64, device='cpu'):
        self.mem_size = mem_size
        self.mem_dim = mem_dim
        self.d_model = d_model
        self.device = device

        # Runtime state (NOT nn.Parameter)
        self.memory = torch.zeros(mem_size, mem_dim, device=device, requires_grad=False)
        self.temporal_positions = torch.zeros(mem_size, device=device, requires_grad=False)
        self.write_ptr = 0
        self.step_count = 0

        # Learnable components (these ARE model parameters)
        self.compress = nn.Linear(d_model, mem_dim)
        self.decompress = nn.Linear(mem_dim, d_model)
        self.read_gate = nn.Linear(d_model, mem_dim)
        self.write_gate = nn.Linear(d_model, 1)
        self.relevance_gate = nn.Linear(d_model + mem_dim, 1)

    def get_parameters(self):
        """Return learnable parameters for optimizer."""
        return list(self.compress.parameters()) + \
               list(self.decompress.parameters()) + \
               list(self.read_gate.parameters()) + \
               list(self.write_gate.parameters()) + \
               list(self.relevance_gate.parameters())

    def read(self, query, top_k=8):
        """
        Read from memory based on query similarity + relevance.
        """
        if query.dim() == 1:
            query = query.unsqueeze(0)

        B = query.shape[0]

        # Compress query to memory space
        q_compressed = self.read_gate(query)  # [B, mem_dim]

        # Expand memory for batch
        mem_expanded = self.memory.unsqueeze(0).expand(B, -1, -1)  # [B, mem_size, mem_dim]

        # Cosine similarity
        q_norm = F.normalize(q_compressed, dim=-1)
        mem_norm = F.normalize(mem_expanded, dim=-1)
        similarities = torch.bmm(q_norm.unsqueeze(1), mem_norm.transpose(1, 2)).squeeze(1)  # [B, mem_size]

        # Recency bias: newer memories get a small boost
        recency = self.temporal_positions / (self.temporal_positions.max() + 1)
        similarities = similarities + 0.1 * recency.unsqueeze(0)

        # Top-k selection
        top_k = min(top_k, self.mem_size)
        top_vals, top_idx = similarities.topk(top_k, dim=-1)  # [B, top_k]

        # Soft attention over retrieved memories
        attn_weights = F.softmax(top_vals, dim=-1)  # [B, top_k]

        # Gather and weight
        retrieved_mem = torch.gather(mem_expanded, 1, top_idx.unsqueeze(-1).expand(-1, -1, self.mem_dim))
        retrieved = (retrieved_mem * attn_weights.unsqueeze(-1)).sum(dim=1)  # [B, mem_dim]

        # Decompress back to model space
        retrieved = self.decompress(retrieved)  # [B, d_model]

        return retrieved, attn_weights

    def write(self, content, force=False):
        """
        Write content to memory.
        """
        # Compute write gate
        gate_value = torch.sigmoid(self.write_gate(content.unsqueeze(0))).item()

        if gate_value < 0.5 and not force:
            return False

        # Compress content (detach to avoid autograd issues)
        compressed = self.compress(content.detach())  # [mem_dim]

        # LRU eviction: find least recently used slot
        if self.write_ptr < self.mem_size:
            slot = self.write_ptr
            self.write_ptr += 1
        else:
            # Evict oldest (lowest temporal position)
            slot = self.temporal_positions.argmin().item()

        # Write with blending (clone to avoid inplace autograd issues)
        write_weight = min(gate_value, 0.9)
        self.memory[slot] = (write_weight * compressed + (1 - write_weight) * self.memory[slot].clone()).detach()

        # Update temporal position
        self.step_count += 1
        self.temporal_positions[slot] = self.step_count

        return True

    def clear(self):
        """Reset memory (for new session)."""
        self.memory.zero_()
        self.temporal_positions.zero_()
        self.write_ptr = 0
        self.step_count = 0

    def get_stats(self):
        """Return memory statistics."""
        return {
            "used_slots": (self.temporal_positions > 0).sum().item(),
            "total_slots": self.mem_size,
            "write_ptr": self.write_ptr,
            "step_count": self.step_count,
        }

    def save(self, path):
        """Save memory state to disk (for cross-session persistence)."""
        state = {
            "memory": self.memory.detach().cpu().numpy().tolist(),
            "temporal_positions": self.temporal_positions.detach().cpu().numpy().tolist(),
            "write_ptr": self.write_ptr,
            "step_count": self.step_count,
        }
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        with open(path, 'w') as f:
            json.dump(state, f)

    def load(self, path):
        """Load memory state from disk."""
        with open(path, 'r') as f:
            state = json.load(f)
        self.memory = torch.tensor(state["memory"], dtype=torch.float32, device=self.device)
        self.temporal_positions = torch.tensor(state["temporal_positions"], dtype=torch.float32, device=self.device)
        self.write_ptr = state["write_ptr"]
        self.step_count = state["step_count"]


# ─── Pillar 3: Reflective Loop ───────────────────────────────────────────
# Contrastive-trained self-correction with 16 learned correction directions.
# Critic predicts which correction (if any) is needed.
# "No correction" option prevents always-on/always-off collapse.

class ReflectiveLoop(nn.Module):
    """
    Self-correction through latent-space traversal.
    - Contrastive training (explicit loss signal for when to correct)
    - Adaptive thresholds (learned, not hardcoded)
    - "No correction" option (prevents always-on/always-off collapse)
    """

    def __init__(self, d_model, num_corrections=16):
        super().__init__()
        self.num_corrections = num_corrections
        self.d_model = d_model

        # Critic: predicts which correction (if any) is needed
        # +1 for "no correction needed"
        self.critic = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, num_corrections + 1)
        )

        # Correction directions (learned)
        self.correction_directions = nn.Parameter(
            torch.randn(num_corrections, d_model) * 0.01
        )

        # Adaptive threshold per correction (learned)
        self.thresholds = nn.Parameter(torch.ones(num_corrections) * 0.5)

        # Confidence scaling
        self.confidence_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, hidden_state, return_correction_id=False):
        if hidden_state.dim() == 3:
            pooled = hidden_state.mean(dim=1)  # [B, D]
        else:
            pooled = hidden_state

        # Compute correction scores [B, num_corrections + 1]
        correction_scores = self.critic(pooled)

        # Last score is "no correction needed"
        no_correction_score = correction_scores[:, -1]
        correction_scores = correction_scores[:, :-1]  # [B, num_corrections]

        # Find best correction
        best_score, best_idx = correction_scores.max(dim=-1)

        # Apply correction only if it beats "no correction" by a margin
        margin = 0.2
        apply_correction = best_score > (no_correction_score + margin)

        corrected_state = hidden_state.clone()
        correction_id = -1

        if apply_correction.any():
            for b in range(hidden_state.shape[0]):
                if apply_correction[b]:
                    correction = self.correction_directions[best_idx[b]]
                    # Scale by confidence (learned threshold)
                    confidence = torch.sigmoid(best_score[b] - self.thresholds[best_idx[b]])
                    scale = torch.abs(self.confidence_scale)

                    if hidden_state.dim() == 3:
                        corrected_state[b] = hidden_state[b] + scale * confidence * correction
                    else:
                        corrected_state[b] = hidden_state[b] + scale * confidence * correction
                    correction_id = best_idx[b].item()

        if return_correction_id:
            return corrected_state, correction_id
        return corrected_state

    def compute_loss(self, hidden_state, is_error, correct_direction=None):
        """
        Contrastive loss for training the critic.
        """
        if hidden_state.dim() == 3:
            pooled = hidden_state.mean(dim=1)
        else:
            pooled = hidden_state

        scores = self.critic(pooled)  # [B, num_corrections + 1]

        # Target: if error -> correct_direction index; if no error -> last index
        targets = torch.where(
            is_error,
            correct_direction,
            torch.full_like(correct_direction, self.num_corrections)
        )

        loss = F.cross_entropy(scores, targets)
        return loss

    def get_correction_stats(self):
        """Return statistics about correction usage."""
        return {
            "num_corrections": self.num_corrections,
            "thresholds_mean": self.thresholds.mean().item(),
            "thresholds_std": self.thresholds.std().item(),
            "confidence_scale": torch.abs(self.confidence_scale).item(),
        }


# ─── Pillar 4: Skill Composition ──────────────────────────────────────────
# 64 composable skills via low-rank perturbations (u @ v^T per skill).
# Skills applied as additive perturbations to hidden state.
# Load balancing loss prevents router collapse.

class SkillComposer(nn.Module):
    """
    Skill Resonance Composition — skills as learned perturbations to latent state.
    64 composable skill vectors at 32K params total.
    Skills are applied as additive perturbations, not weight modifications.
    """

    SKILL_NAMES = [
        "MATHEMATICS", "LOGIC", "WRITING", "CODE", "ANALYSIS",
        "CREATIVITY", "REASONING", "RECALL", "SUMMARY", "PLANNING",
        "DEBUGGING", "EXPLANATION", "COMPARISON", "CLASSIFICATION",
        "GENERATION", "TRANSLATION", "OPTIMIZATION", "VERIFICATION",
        "RESEARCH", "SYNTHESIS", "DECOMPOSITION", "ABSTRACTION",
        "PATTERN_MATCH", "CAUSAL_REASONING", "TEMPORAL", "SPATIAL",
        "EMOTIONAL", "PERSUASION", "TEACHING", "QUESTIONING",
        "HYPOTHESIS", "TESTING", "MEASUREMENT", "ESTIMATION",
        "PRIORITIZATION", "SEQUENCING", "ITERATION", "REFINEMENT",
        "AGGREGATION", "FILTERING", "TRANSFORMATION", "MAPPING",
        "VALIDATION", "CRITIQUE", "INTUITION", "HEURISTIC",
        "FORMALIZATION", "CONTEXTUALIZATION", "COMPRESSION", "EXPANSION",
        "FOCUS", "DIVERGENCE", "CONVERGENCE", "BRIDGING",
        "NARRATIVE", "ARGUMENTATION", "DEFINITION", "EXEMPLIFICATION",
        "QUANTIFICATION", "QUALITATIVE", "SYSTEMS_THINKING", "META_COGNITION",
        "ERROR_CORRECTION", "KNOWLEDGE_TRANSFER", "ADAPTATION", "INTEGRATION",
    ]

    def __init__(self, d_model, num_skills=64, skill_rank=8, top_k=4):
        super().__init__()
        self.num_skills = num_skills
        self.skill_rank = skill_rank
        self.top_k = top_k
        self.d_model = d_model

        # Low-rank skill vectors: u @ v^T per skill
        # Total params: num_skills * d_model * skill_rank * 2
        self.skill_u = nn.Parameter(torch.randn(num_skills, d_model, skill_rank) * 0.01)
        self.skill_v = nn.Parameter(torch.randn(num_skills, skill_rank, d_model) * 0.01)

        # Router: decides which skills to activate
        self.router = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Linear(d_model // 4, num_skills)
        )

        # Skill scaling (learned)
        self.skill_scale = nn.Parameter(torch.ones(num_skills) * 0.01)

    def forward(self, x, return_skill_info=False):
        B, T, D = x.shape

        # Compute skill weights from mean pooled representation
        pooled = x.mean(dim=1)  # [B, D]
        skill_logits = self.router(pooled)  # [B, num_skills]
        skill_weights = F.softmax(skill_logits, dim=-1)  # [B, num_skills]

        # Load balancing: encourage uniform usage across skills
        if self.training:
            avg_usage = skill_weights.mean(dim=0)  # [num_skills]
            self._load_balance_loss = (avg_usage.var() * 10.0)
        else:
            self._load_balance_loss = torch.tensor(0.0)

        # Select top-k skills
        top_k = min(self.top_k, self.num_skills)
        top_weights, top_indices = skill_weights.topk(top_k, dim=-1)  # [B, top_k]
        top_weights = top_weights / (top_weights.sum(dim=-1, keepdim=True) + 1e-8)

        # Compose skill perturbations
        perturbation = torch.zeros_like(x)  # [B, T, D]

        for k in range(self.top_k):
            skill_idx = top_indices[:, k]  # [B]
            weight = top_weights[:, k]  # [B]

            # Gather skill parameters for this batch
            u = self.skill_u[skill_idx]  # [B, D, rank]
            v = self.skill_v[skill_idx]  # [B, rank, D]
            scale = torch.abs(self.skill_scale[skill_idx])  # [B]

            # Compute low-rank perturbation: x @ v^T @ u^T
            x_v = torch.bmm(x, v.transpose(1, 2))  # [B, T, rank]
            skill_perturbation = torch.bmm(x_v, u.transpose(1, 2))  # [B, T, D]

            # Apply with weight and scale
            perturbation += weight.unsqueeze(1).unsqueeze(-1) * scale.unsqueeze(1).unsqueeze(-1) * skill_perturbation

        output = x + perturbation

        if return_skill_info:
            active_skills = []
            for b in range(B):
                batch_skills = []
                for k in range(self.top_k):
                    idx = top_indices[b, k].item()
                    batch_skills.append({
                        "name": self.SKILL_NAMES[idx] if idx < len(self.SKILL_NAMES) else f"skill_{idx}",
                        "weight": top_weights[b, k].item(),
                        "scale": torch.abs(self.skill_scale[idx]).item(),
                    })
                active_skills.append(batch_skills)

            return output, {
                "active_skills": active_skills,
                "skill_weights": skill_weights,
                "top_indices": top_indices,
            }

        return output

    def get_skill_parameters(self):
        """Return all skill-related parameters."""
        return [self.skill_u, self.skill_v, self.skill_scale] + \
               list(self.router.parameters())


# ─── Memory Layer (wraps EpisodicMemory for hook integration) ─────────────

class CRNMemoryLayer(nn.Module):
    """Episodic memory layer that reads/writes to memory."""

    def __init__(self, d_model, mem_size=512, mem_dim=128, device='cpu'):
        super().__init__()
        self.d_model = d_model

        # Episodic Memory (runtime state)
        self.memory = EpisodicMemory(d_model, mem_size, mem_dim, device)

        # Cast memory learnable modules to bfloat16 and move to device
        for mod in [self.memory.compress, self.memory.decompress,
                    self.memory.read_gate, self.memory.write_gate, self.memory.relevance_gate]:
            mod.to(device, dtype=torch.bfloat16)

        # Cast memory runtime state to bfloat16
        self.memory.memory = self.memory.memory.to(dtype=torch.bfloat16)
        self.memory.temporal_positions = self.memory.temporal_positions.to(dtype=torch.bfloat16)

        # Read/write gates
        self.read_gate = nn.Linear(d_model, d_model, dtype=torch.bfloat16)
        self.write_gate = nn.Linear(d_model, 1, dtype=torch.bfloat16)
        self.blend = nn.Parameter(torch.tensor(0.1, dtype=torch.bfloat16))

    def read(self, hidden_states):
        """Read from memory and blend with hidden states."""
        if self.memory.temporal_positions.sum() == 0:
            return hidden_states

        B, T, D = hidden_states.shape
        query = hidden_states.mean(dim=1)  # [B, D]
        query_proj = self.read_gate(query)
        retrieved, _ = self.memory.read(query_proj, top_k=8)

        # Blend with hidden states
        blend = torch.sigmoid(self.blend)
        hidden_states = hidden_states + blend * retrieved.unsqueeze(1)

        return hidden_states

    def write(self, hidden_states):
        """Write final hidden state to memory."""
        if self.training:
            content = hidden_states[:, -1, :].mean(dim=0)  # [D]
            self.memory.write(content, force=False)

    def get_parameters(self):
        """Get learnable parameters."""
        params = self.memory.get_parameters()
        params += list(self.read_gate.parameters())
        params += list(self.write_gate.parameters())
        params.append(self.blend)
        return params

    def save(self, path):
        self.memory.save(path)

    def load(self, path):
        self.memory.load(path)


print('CRN components: ResonanceAttention, EpisodicMemory, ReflectiveLoop, SkillComposer')

In [ ]:
# Cell 4: Student Model with Full CRN via 46 Hooks

class PrajnaStudent(nn.Module):
    def __init__(self, device='cuda'):
        super().__init__()
        self.device = device
        print('Loading E2B student...')
        self.tok = AutoTokenizer.from_pretrained('google/gemma-4-E2B')
        self.model = AutoModelForCausalLM.from_pretrained(
            'google/gemma-4-E2B', dtype=torch.bfloat16, device_map='auto'
        )

        # Full CRN components
        self.mem = CRNMemoryLayer(d_model=1536, mem_size=512, mem_dim=128, device=device)
        self.reflection = ReflectiveLoop(d_model=1536, num_corrections=16)
        self.skills = SkillComposer(d_model=1536, num_skills=64, skill_rank=8, top_k=4)
        self.resonance = ResonanceAttention(d_model=1536, num_heads=4, num_frequencies=16, top_k=4)

        # Cast CRN to bfloat16 and move to device
        self.reflection = self.reflection.to(device, dtype=torch.bfloat16)
        self.skills = self.skills.to(device, dtype=torch.bfloat16)
        self.resonance = self.resonance.to(device, dtype=torch.bfloat16)

        self.vocab = 262144

        # Freeze base model
        for p in self.model.parameters():
            p.requires_grad = False

        self._hooks = []
        layers = self.model.model.language_model.layers
        mid = len(layers) // 2

        # Pillar 2: Memory read hook at midpoint (before layer 17)
        self._hooks.append(layers[mid].register_forward_pre_hook(
            lambda m, i: (self.mem.read(i[0].to(self.device)).to(i[0].device),) + i[1:]
        ))

        # Pillar 2: Memory write hook after final layer
        def mem_write(m, i, o):
            h = o[0] if isinstance(o, tuple) else o
            self.mem.write(h.to(self.device))
            return o
        self._hooks.append(layers[-1].register_forward_hook(mem_write))

        # Pillar 3: Reflective loop hook after each layer
        for l in layers:
            def ref_hook(m, i, o):
                h = o[0] if isinstance(o, tuple) else o
                c = self.reflection(h.to(self.device)).to(h.device)
                return (c,) + o[1:] if isinstance(o, tuple) else c
            self._hooks.append(l.register_forward_hook(ref_hook))

        # Pillar 4: Skill composition hook after every 4th layer
        for i, l in enumerate(layers):
            if i % 4 == 0:
                def skill_hook(m, inp, o):
                    h = o[0] if isinstance(o, tuple) else o
                    s = self.skills(h.to(self.device)).to(h.device)
                    return (s,) + o[1:] if isinstance(o, tuple) else s
                self._hooks.append(l.register_forward_hook(skill_hook))

        # Pillar 1: Resonance attention hook on first half of layers
        for i, l in enumerate(layers):
            if i < mid:
                def res_hook(m, inp, o):
                    h = o[0] if isinstance(o, tuple) else o
                    r = self.resonance(h.to(self.device)).to(h.device)
                    return (r,) + o[1:] if isinstance(o, tuple) else r
                self._hooks.append(l.register_forward_hook(res_hook))

        params = self.get_params()
        print(f'CRN: {sum(p.numel() for p in params):,} params, {len(self._hooks)} hooks')
        print(f'  Memory: {self.mem.memory.get_stats()}')

    def forward(self, input_ids, labels=None):
        out = self.model(input_ids=input_ids)
        loss = None
        if labels is not None:
            # Main distillation loss (cross-entropy)
            ce_loss = F.cross_entropy(
                out.logits[:,:-1].reshape(-1, self.vocab),
                labels[:,1:].reshape(-1), ignore_index=-100
            )

            # Pillar 3: Contrastive loss for reflective loop
            # Simulate error detection: compare logits to labels
            with torch.no_grad():
                preds = out.logits[:,:-1].argmax(dim=-1)
                targets = labels[:,1:]
                is_error = (preds != targets) & (targets != -100)

            # Get hidden states from last layer for contrastive loss
            # Use the model's hidden states (accessed via hooks)
            # For contrastive loss, we use the output logits as a proxy
            pooled = out.logits.mean(dim=1)  # [B, vocab]

            # Contrastive loss: train critic to detect errors
            contrastive_loss = torch.tensor(0.0, device=self.device)
            if is_error.any():
                # Create fake hidden states from logits for contrastive training
                fake_hidden = pooled.unsqueeze(1)  # [B, 1, vocab]
                # Use a small projection to match d_model
                if not hasattr(self, '_proj'):
                    self._proj = nn.Linear(self.vocab, 1536, dtype=torch.bfloat16).to(self.device)
                proj_hidden = self._proj(fake_hidden)  # [B, 1, 1536]

                # Random error directions for contrastive training
                B = is_error.shape[0]
                correct_dir = torch.randint(0, 16, (B,), device=self.device)
                contrastive_loss = self.reflection.compute_loss(
                    proj_hidden, is_error, correct_dir
                )

            # Combined loss
            loss = ce_loss + 0.1 * contrastive_loss

        return {'loss': loss, 'ce_loss': ce_loss.item() if labels is not None else 0}

    def get_params(self):
        params = []
        params += self.mem.get_parameters()
        params += list(self.reflection.parameters())
        params += list(self.skills.parameters())
        params += list(self.resonance.parameters())
        if hasattr(self, '_proj'):
            params += list(self._proj.parameters())
        return params

    def save_memory(self, path):
        self.mem.save(path)

    def load_memory(self, path):
        self.mem.load(path)

    def cleanup(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

print('Student model defined with full CRN')

In [ ]:
# Cell 5: Dataset
from torch.utils.data import Dataset, DataLoader
import json

class DistillationDataset(Dataset):
    def __init__(self, data_dir, tokenizer, max_length=512):
        self.samples = []
        self.tok = tokenizer
        self.ml = max_length
        if os.path.exists(data_dir):
            for f in sorted(os.listdir(data_dir)):
                if f.endswith('.json'):
                    with open(os.path.join(data_dir, f)) as fh:
                        self.samples.extend(json.load(fh))
        print(f'Dataset: {len(self.samples)} samples')
    def __len__(self):
        return max(len(self.samples), 1)
    def __getitem__(self, i):
        if not self.samples:
            d = torch.zeros(self.ml, dtype=torch.long)
            return {'input_ids': d, 'labels': d.clone()}
        s = self.samples[i % len(self.samples)]
        text = f"{s.get('prompt','')}\n\n{s.get('response','')}"
        enc = self.tok(text, truncation=True, max_length=self.ml,
                       padding='max_length', return_tensors='pt')
        ids = enc['input_ids'].squeeze()
        labels = ids.clone()
        labels[enc['attention_mask'].squeeze() == 0] = -100
        return {'input_ids': ids, 'labels': labels}

print('Dataset defined')

In [ ]:
# Cell 6: Load E4B Teacher (4-bit)
from transformers import BitsAndBytesConfig

print('Loading E4B teacher (4-bit)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,
)

teacher = AutoModelForCausalLM.from_pretrained(
    'google/gemma-4-E4B',
    quantization_config=bnb_config,
    device_map='auto',
    offload_folder='offload',
)
teacher_tok = AutoTokenizer.from_pretrained('google/gemma-4-E4B')
for p in teacher.parameters():
    p.requires_grad = False

print(f'Teacher loaded: {sum(p.numel() for p in teacher.parameters()):,} params')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 7: Generate Training Data
import time

data_dir = '/content/drive/MyDrive/prajna/data'
data_file = os.path.join(data_dir, 'teacher_data.json')
partial_file = data_file + '.partial'

n_samples = 3000
seqs_per_call = 6
prompts = [
    # Reasoning & Analysis
    'Analyze the pros and cons of remote work vs office work',
    'Compare and contrast democracy and authoritarianism',
    'What are the logical steps to solve a Sudoku puzzle?',
    'Explain the causal chain from deforestation to climate change',
    'Evaluate: All birds can fly. Penguins are birds. Therefore penguins can fly.',
    # Math & Science
    'Solve: A train travels 120km in 2h, then 180km in 3h. What is average speed?',
    'Explain why the sky appears red at sunset using physics',
    'What is the derivative of x^3 + 2x^2 - 5x + 7?',
    'Describe the water cycle and how each stage works',
    'Explain how CRISPR gene editing works at a molecular level',
    # Coding
    'Write a Python function to find all primes up to n using Sieve of Eratosthenes',
    'Debug this code: def fib(n): return fib(n-1) + fib(n-2)',
    'Explain the difference between a stack and a queue with examples',
    'Write SQL to find top 3 customers by total spending from an orders table',
    'How would you design a URL shortener like bit.ly?',
    # Creative Writing
    'Write a short story about a robot discovering emotions',
    'Compose a poem about the passage of time',
    'Write a persuasive essay arguing for universal basic income',
    'Create a dialogue between two scientists debating dark matter',
    'Write a product description for a device that translates animal thoughts',
    # Explanation & Teaching
    'Explain quantum entanglement to a 10-year-old',
    'How does a neural network learn to recognize cats in images?',
    'Explain the stock market crash of 2008 in simple terms',
    'What is blockchain and how does it ensure trust?',
    'Teach me about the Renaissance in 5 key points',
    # Problem Solving
    'I have eggs, bread, and cheese. What can I cook in 10 minutes?',
    'My WiFi drops every 30 minutes. Walk me through troubleshooting',
    'Plan a surprise party for 20 people with $500 budget',
    'How should I prioritize urgent vs important tasks?',
    'My code is O(n^2). How can I optimize to O(n log n)?',
    # Memory & Context
    'Remember: My birthday is March 15, I prefer dark mode, I like Python. What is my birthday?',
    'Earlier I said I work at NexGen. What company do I work at?',
    'I mentioned preferring tea over coffee. What do I prefer?',
    'Recall the three laws of robotics I described',
    'What was the first fact I asked you to remember?',
    # Multi-step Tasks
    'Walk me through building a REST API: planning, design, implementation, testing',
    'Explain publishing a research paper: submission to publication',
    'How to train an ML model from raw data to deployment step by step',
    'How to migrate a monolith to microservices? Give a phased approach',
    'Outline building a mobile app: ideation to launch',
    # Ethics & Philosophy
    'Is it ethical to use AI for hiring? Present both sides',
    'Should social media be responsible for misinformation?',
    'What are ethical implications of autonomous weapons?',
    'Examine the trolley problem with modern variations',
    'Should genetic engineering of humans be allowed?',
    # Business & Strategy
    'Explain B2B vs B2C marketing strategies',
    'What metrics should a SaaS startup track?',
    'How would you enter a market dominated by a monopoly?',
    'Analyze Netflix business model',
    'What is product-market fit and how do you measure it?',
    # Language & Communication
    'Rewrite for formal email: Hey can you send me that report ASAP?',
    'Explain effect vs affect with examples',
    'How to negotiate a salary offer? Give specific phrases',
    'Translate into layman terms: algorithmic trading',
    'Write a LinkedIn summary for a software engineer',
]
n_calls = n_samples // seqs_per_call

def generate_batch(samples_list, start_i, n_batches, label):
    for i in range(n_batches):
        p = prompts[(start_i + i) % len(prompts)]
        inputs = teacher_tok(p, return_tensors='pt').to(teacher.device)
        with torch.no_grad():
            out = teacher.generate(
                **inputs,
                max_new_tokens=80, temperature=0.8, do_sample=True,
                num_return_sequences=seqs_per_call, pad_token_id=teacher_tok.pad_token_id or teacher_tok.eos_token_id
            )
        for j in range(seqs_per_call):
            response = teacher_tok.decode(out[j], skip_special_tokens=True)
            samples_list.append({'prompt': p, 'response': response})
        if (i+1) % 50 == 0:
            elapsed = time.time() - t0
            rate = (i+1) * seqs_per_call / elapsed
            total_done = len(samples_list)
            remaining = n_batches - i - 1
            eta = remaining * seqs_per_call / rate
            print(f'  {total_done}/{n_samples} ({elapsed:.0f}s, {rate:.1f} samp/s, ETA: {eta:.0f}s)')
            with open(partial_file, 'w') as f:
                json.dump(samples_list, f)

if not os.path.exists(data_file):
    print(f'Generating {n_samples} samples from E4B ({n_calls} calls × {seqs_per_call} sequences)...')
    samples = []
    t0 = time.time()
    generate_batch(samples, 0, n_calls, 'fresh')
    with open(data_file, 'w') as f:
        json.dump(samples, f, indent=2)
    if os.path.exists(partial_file):
        os.remove(partial_file)
    print(f'Saved {len(samples)} samples')
elif os.path.exists(partial_file):
    print('Resuming from partial...')
    with open(partial_file) as f:
        samples = json.load(f)
    n_remaining = n_samples - len(samples)
    n_calls_rem = n_remaining // seqs_per_call
    print(f'  Found {len(samples)} samples, generating {n_remaining} more')
    t0 = time.time()
    generate_batch(samples, len(samples) // seqs_per_call, n_calls_rem, 'resume')
    with open(data_file, 'w') as f:
        json.dump(samples, f, indent=2)
    if os.path.exists(partial_file):
        os.remove(partial_file)
    print(f'Saved {len(samples)} samples')
else:
    print(f'Data exists: {data_file}')

In [ ]:
# Cell 8: Free teacher memory
del teacher
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 9: Create Student
student = PrajnaStudent(device='cuda')
opt = torch.optim.AdamW(student.get_params(), lr=2e-4)
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Print CRN component stats
print(f'\nCRN Component Stats:')
print(f'  Memory: {student.mem.memory.get_stats()}')
print(f'  Reflection: {student.reflection.get_correction_stats()}')
print(f'  Skills: {student.skills.num_skills} skills, rank={student.skills.skill_rank}')
print(f'  Resonance: {student.resonance.num_frequencies} frequencies, top_k={student.resonance.top_k}')

In [ ]:
# Cell 10: Training with Memory Persistence
import time

print('='*60)
print('TRAINING STARTED — Full CRN Architecture')
print('='*60)

dataset = DistillationDataset(data_dir, student.tok)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

losses = []
ce_losses = []
t_start = time.time()

for epoch in range(5):
    print(f'\nEpoch {epoch+1}/5')
    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to('cuda')
        labels = batch['labels'].to('cuda')
        student.train()
        out = student(input_ids, labels)
        loss = out['loss']
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.get_params(), 1.0)
        opt.step()
        opt.zero_grad()
        losses.append(loss.item())
        ce_losses.append(out.get('ce_loss', 0))
        if len(losses) % 10 == 0:
            avg = sum(losses[-10:]) / 10
            vram = torch.cuda.memory_allocated()/1e9
            mem_stats = student.mem.memory.get_stats()
            print(f'  Step {len(losses):5d} | Loss: {loss.item():.4f} | Avg: {avg:.4f} | VRAM: {vram:.1f}GB | Mem slots: {mem_stats["used_slots"]}/{mem_stats["total_slots"]}')
        if len(losses) % 500 == 0 and len(losses) > 0:
            ckpt = {
                'step': len(losses),
                'crn': {k: v for k, v in student.state_dict().items() if not k.startswith('model')},
                'loss': sum(losses[-50:]) / len(losses[-50:]),
                'memory_stats': student.mem.memory.get_stats(),
            }
            torch.save(ckpt, f'/content/drive/MyDrive/prajna/checkpoints/ckpt_{len(losses)}.pt')
            # Save memory state separately
            student.save_memory(f'/content/drive/MyDrive/prajna/checkpoints/memory_{len(losses)}.json')
            print(f'  Checkpoint + memory saved!')

final_loss = sum(losses[-50:]) / len(losses[-50:]) if losses else 0
ckpt = {
    'step': len(losses),
    'crn': {k: v for k, v in student.state_dict().items() if not k.startswith('model')},
    'loss': final_loss,
    'memory_stats': student.mem.memory.get_stats(),
    'correction_stats': student.reflection.get_correction_stats(),
}
torch.save(ckpt, '/content/drive/MyDrive/prajna/checkpoints/best.pt')
student.save_memory('/content/drive/MyDrive/prajna/checkpoints/memory_best.json')

elapsed = (time.time() - t_start) / 60
print('\n' + '='*60)
print('TRAINING COMPLETE — Full CRN Architecture')
print(f'Steps: {len(losses)}')
print(f'Final loss: {final_loss:.4f}')
print(f'Memory stats: {student.mem.memory.get_stats()}')
print(f'Correction stats: {student.reflection.get_correction_stats()}')
print(f'Time: {elapsed:.1f} min')
print('='*60)

In [ ]:
# Cell 11: Verify & Cleanup
ckpt = torch.load('/content/drive/MyDrive/prajna/checkpoints/best.pt', weights_only=False)
print(f'Steps: {ckpt["step"]}')
print(f'Loss: {ckpt["loss"]:.4f}')
print(f'CRN keys: {list(ckpt["crn"].keys())[:5]}')
print(f'Memory stats: {ckpt.get("memory_stats", {})}')
print(f'Correction stats: {ckpt.get("correction_stats", {})}')

print('\nCheckpoints:')
for f in sorted(os.listdir('/content/drive/MyDrive/prajna/checkpoints/')):
    print(f'  {f}')

student.cleanup()
print('\nDone!')